# SignSense - Notebook 2: Alphabet A-Z (Dense MLP)

Trains on **real ASL images** from Kaggle with **data augmentation**.

### Setup: Upload kaggle.json FIRST
1. kaggle.com > Account > API > Create New Token
2. Upload `kaggle.json` to Colab Files panel
3. Run All

In [ ]:
# CELL 1: INSTALL
!pip install -q mediapipe tensorflowjs scikit-learn matplotlib opencv-python-headless kaggle
print('Install done.')

In [ ]:
# CELL 2: IMPORTS
import os, cv2, json, shutil, re, urllib.request
import numpy as np
import mediapipe as mp
import tensorflow as tf
import tensorflowjs as tfjs
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
print(f'TF {tf.__version__}, MediaPipe {mp.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# CELL 3: CONFIG
CLASSES         = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
NUM_CLASSES     = 26
HAND_DIM        = 63
SAMPLES_PER_CLS = 2000   # Extract up to 2000 real landmarks per class
AUG_FACTOR      = 5      # Generate 5x augmented copies per real sample
DATA_DIR        = 'data/alpha_landmarks'
MODEL_SAVE_PATH = 'models/alpha_model_keras'
TFJS_SAVE_PATH  = 'models/alpha_model_tfjs'
print(f'{NUM_CLASSES} classes, {SAMPLES_PER_CLS} raw + {AUG_FACTOR}x augmented')

In [ ]:
# CELL 4: DOWNLOAD MEDIAPIPE HAND MODEL
HAND_MODEL_PATH = 'hand_landmarker.task'
if not os.path.exists(HAND_MODEL_PATH):
    url = ('https://storage.googleapis.com/mediapipe-models/'
           'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task')
    urllib.request.urlretrieve(url, HAND_MODEL_PATH)
print(f'Hand model: {os.path.getsize(HAND_MODEL_PATH)/1e6:.1f} MB')

In [ ]:
# CELL 5: MEDIAPIPE SETUP
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

def make_detector(num_hands=1):
    return HandLandmarker.create_from_options(HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=HAND_MODEL_PATH),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=num_hands,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5))

def extract_hand(result):
    """Extract normalized 63-float vector. Matches script.js extractFeatures()."""
    if not result.hand_landmarks: return None
    lms = result.hand_landmarks[0]
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z
    coords = np.array([[l.x-wx, l.y-wy, l.z-wz] for l in lms]).flatten()
    mx = np.max(np.abs(coords))
    return (coords / (mx if mx else 1.0)).astype(np.float32)

print('MediaPipe ready')

In [ ]:
# CELL 6: DOWNLOAD KAGGLE ASL ALPHABET
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d grassknoted/asl-alphabet -p data/ --unzip
print('Dataset downloaded!')

In [ ]:
# CELL 7: EXTRACT LANDMARKS FROM IMAGES
os.makedirs(DATA_DIR, exist_ok=True)
total_extracted = 0

with make_detector(num_hands=1) as det:
    for cls in CLASSES:
        candidates = [
            f'data/asl_alphabet_train/asl_alphabet_train/{cls}',
            f'data/asl_alphabet_train/{cls}',
            f'data/{cls}'
        ]
        cls_dir = next((p for p in candidates if os.path.exists(p)), None)
        if cls_dir is None:
            print(f'  SKIP {cls}: folder not found'); continue

        imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        np.random.shuffle(imgs)

        lms_list, count = [], 0
        for img_file in imgs:
            if count >= SAMPLES_PER_CLS: break
            img = cv2.imread(os.path.join(cls_dir, img_file))
            if img is None: continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            lm = extract_hand(det.detect(mp_img))
            if lm is not None:
                lms_list.append(lm); count += 1

        if lms_list:
            np.save(os.path.join(DATA_DIR, f'{cls}.npy'),
                    np.array(lms_list, dtype=np.float32))
            total_extracted += len(lms_list)
            print(f'  {cls}: {len(lms_list)} landmarks')
        else:
            print(f'  {cls}: 0 hands detected!')

print(f'\nTotal extracted: {total_extracted} landmarks')

In [ ]:
# CELL 8: DATA AUGMENTATION
# Augments landmark vectors to improve robustness to webcam conditions

def augment_landmarks(data, factor=5, seed=42):
    """
    Augment normalized landmark vectors with:
    1. Gaussian noise (simulates MediaPipe jitter)
    2. Random scaling (simulates distance variation)
    3. Random 2D rotation on (x,y) pairs (simulates wrist tilt)
    4. X-axis flip (simulates left/right hand or camera mirror)
    """
    rng = np.random.default_rng(seed)
    augmented = [data.copy()]  # include originals

    for _ in range(factor):
        batch = data.copy()
        n = len(batch)

        # 1. Gaussian noise (small, simulates landmark jitter)
        noise = rng.normal(0, 0.02, batch.shape).astype(np.float32)
        batch = batch + noise

        # 2. Random per-sample scaling (0.85 to 1.15)
        scales = rng.uniform(0.85, 1.15, (n, 1)).astype(np.float32)
        batch = batch * scales

        # 3. Random 2D rotation on (x,y) pairs (-15 to +15 degrees)
        angles = rng.uniform(-15, 15, n) * np.pi / 180
        cos_a = np.cos(angles).astype(np.float32)
        sin_a = np.sin(angles).astype(np.float32)
        rotated = batch.copy()
        for lm in range(21):  # 21 landmarks
            xi = lm * 3      # x index
            yi = lm * 3 + 1  # y index
            x_old = batch[:, xi]
            y_old = batch[:, yi]
            rotated[:, xi] = cos_a * x_old - sin_a * y_old
            rotated[:, yi] = sin_a * x_old + cos_a * y_old
        batch = rotated

        # 4. Random x-flip (50% chance per sample)
        flip_mask = rng.random(n) < 0.5
        for lm in range(21):
            xi = lm * 3
            batch[flip_mask, xi] *= -1

        # Re-normalize: divide by max absolute value per sample
        max_vals = np.max(np.abs(batch), axis=1, keepdims=True)
        max_vals[max_vals == 0] = 1.0
        batch = batch / max_vals

        augmented.append(batch.astype(np.float32))

    return np.vstack(augmented)

# Apply augmentation to all classes
X_all, y_all = [], []
for i, cls in enumerate(CLASSES):
    path = os.path.join(DATA_DIR, f'{cls}.npy')
    if not os.path.exists(path):
        print(f'  Missing: {cls}'); continue
    raw = np.load(path)
    aug = augment_landmarks(raw, factor=AUG_FACTOR, seed=42+i)
    X_all.append(aug)
    y_all.extend([i] * len(aug))
    print(f'  {cls}: {len(raw)} raw -> {len(aug)} augmented')

X = np.vstack(X_all).astype(np.float32)
y = np.array(y_all, dtype=np.int32)
print(f'\nTotal dataset: X={X.shape}, y={y.shape}')
u, c = np.unique(y, return_counts=True)
print(f'Samples/class: min={c.min()}, max={c.max()}')

In [ ]:
# CELL 9: SPLIT
y_cat = to_categorical(y, NUM_CLASSES)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y_cat, test_size=0.15,
                                              random_state=42, stratify=y)
X_v, X_te, y_v, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50,
                                          random_state=42)
print(f'Train={X_tr.shape[0]}  Val={X_v.shape[0]}  Test={X_te.shape[0]}')

In [ ]:
# CELL 10: BUILD MODEL (larger for augmented data)
model = Sequential([
    Input(shape=(HAND_DIM,)),
    Dense(1024, activation='relu'), BatchNormalization(), Dropout(0.5),
    Dense(512, activation='relu'), BatchNormalization(), Dropout(0.4),
    Dense(256, activation='relu'), BatchNormalization(), Dropout(0.3),
    Dense(128, activation='relu'), BatchNormalization(), Dropout(0.2),
    Dense(64, activation='relu'), Dropout(0.2),
    Dense(NUM_CLASSES, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# CELL 11: TRAIN
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
cbs = [
    EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_SAVE_PATH,'best.h5'),
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]
history = model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
                    epochs=200, batch_size=128, callbacks=cbs, verbose=1)

In [ ]:
# CELL 12: EVALUATE
loss, acc = model.evaluate(X_te, y_te, verbose=0)
print(f'Test Accuracy: {acc*100:.2f}%  Loss: {loss:.4f}')
yp = np.argmax(model.predict(X_te), axis=1)
yt = np.argmax(y_te, axis=1)
print(classification_report(yt, yp, target_names=CLASSES))

fig, (a1,a2) = plt.subplots(1,2,figsize=(12,4))
a1.plot(history.history['accuracy'], label='Train')
a1.plot(history.history['val_accuracy'], label='Val')
a1.set_title('Accuracy'); a1.legend(); a1.set_ylim(0,1)
a2.plot(history.history['loss'], label='Train')
a2.plot(history.history['val_loss'], label='Val')
a2.set_title('Loss'); a2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# CELL 13: EXPORT TO TF.JS (with Keras 3 compatibility fix)
os.makedirs(TFJS_SAVE_PATH, exist_ok=True)
tfjs.converters.save_keras_model(model, TFJS_SAVE_PATH)

# Fix Keras 3 JSON for TF.js compatibility
model_json_path = os.path.join(TFJS_SAVE_PATH, 'model.json')
with open(model_json_path) as f:
    mj = json.load(f)

def fix_dtype(obj):
    if isinstance(obj, dict):
        if obj.get('class_name') == 'DTypePolicy':
            return obj.get('config',{}).get('name','float32')
        return {k: fix_dtype(v) for k,v in obj.items()}
    if isinstance(obj, list): return [fix_dtype(v) for v in obj]
    return obj

mj = fix_dtype(mj)

for layer in mj.get('modelTopology',{}).get('model_config',{}).get('config',{}).get('layers',[]):
    cfg = layer.get('config',{})
    if layer['class_name'] == 'InputLayer':
        if 'batch_shape' in cfg:
            cfg['batch_input_shape'] = cfg.pop('batch_shape')
        cfg.pop('optional', None)
    for k,v in list(cfg.items()):
        if isinstance(v, dict) and 'class_name' in v:
            v.pop('module',None); v.pop('registered_name',None)
            if 'config' in v: v['config'].pop('seed',None)
    for rm in ['quantization_config','zero_output_for_mask','synchronized','seed']:
        cfg.pop(rm, None)

# Fix weight name prefixes (Keras 3 adds 'sequential/' prefix)
import re
for g in mj.get('weightsManifest',[]):
    for w in g.get('weights',[]):
        w['name'] = re.sub(r'^sequential(_\d+)?/', '', w['name'])

with open(model_json_path, 'w') as f:
    json.dump(mj, f)

# Write metadata
meta = {'classes':CLASSES,'input_dim':HAND_DIM,
        'num_classes':NUM_CLASSES,'test_accuracy':float(acc)}
with open(os.path.join(TFJS_SAVE_PATH,'metadata.json'),'w') as f:
    json.dump(meta, f, indent=2)

print('Exported TF.js files:')
for fn in sorted(os.listdir(TFJS_SAVE_PATH)):
    sz = os.path.getsize(os.path.join(TFJS_SAVE_PATH, fn))
    print(f'  {fn}  {sz/1024:.1f} KB')

In [ ]:
# CELL 14: ZIP & DOWNLOAD
shutil.make_archive('alpha_model','zip', TFJS_SAVE_PATH)
try:
    from google.colab import files; files.download('alpha_model.zip')
    print('Download triggered!')
except ImportError:
    print('Find alpha_model.zip in the file browser.')
print('Done! Unzip -> copy to project/models/alpha_model/')